<a href="https://colab.research.google.com/github/1900690/fisheye-distortion-correction/blob/main/%E3%82%BF%E3%82%A4%E3%83%A0%E3%83%A9%E3%83%97%E3%82%B9%E7%94%A8%E3%81%AE%E5%8B%95%E7%94%BB%E5%88%87%E3%82%8A%E5%8F%96%E3%82%8A%E3%81%A8%E8%A3%9C%E6%AD%A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#タイムラプス用動画の切り取りと補正を行うプログラム

In [ ]:
#@title 1. AVI動画のアップロード
import os
import cv2
import matplotlib.pyplot as plt
from google.colab import files

# フォルダの初期化（以前のデータをクリア）
for dir_name in ['/content/extracted_images', '/content/corrected_images']:
    !rm -rf {dir_name}
    os.makedirs(dir_name, exist_ok=True)

print("AVI動画ファイルを選択してください。")
uploaded = files.upload()

if uploaded:
    video_filename = list(uploaded.keys())[0]
    print(f"アップロード完了: {video_filename}")

    # 動画の最初のフレームを表示
    cap = cv2.VideoCapture(video_filename)
    if cap.isOpened():
        ret, first_frame = cap.read()
        if ret:
            print("\n【動画の最初のフレーム（確認用）】")
            plt.figure(figsize=(8, 6))
            plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()
        else:
            print("動画の最初のフレームを読み込めませんでした。")
        cap.release()
    else:
        print("動画ファイルを開けませんでした。")
else:
    print("アップロードがキャンセルされました。")

In [ ]:
#@title 2. 特定タイミングの画像抽出

import cv2
import os
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import re

#@markdown **【タイムラプス（元動画）の設定】**
動画の撮影開始日時 = "2026/06/15 16:09:44" #@param {type:"string"}
撮影間隔_分秒 = "10分 0秒" #@param {type:"string"}

#@markdown **【抽出の範囲設定】**
#@markdown ※空欄にすると「最初から最後まで」として処理されます。
抽出開始_時間_分_秒 = "" #@param {type:"string"}
抽出終了_時間_分_秒 = "" #@param {type:"string"}

#@markdown **【抽出間隔】**
抽出間隔_分_秒 = "1時間 0分 0秒" #@param {type:"string"}

def parse_time_str(time_str):
    """'H時間 M分 S秒' 形式の文字列を秒数に変換する"""
    if not time_str or time_str.strip() == "":
        return 0
    seconds = 0
    h = re.search(r'(\d+)時間', time_str)
    m = re.search(r'(\d+)分', time_str)
    s = re.search(r'(\d+)秒', time_str)
    if h: seconds += int(h.group(1)) * 3600
    if m: seconds += int(m.group(1)) * 60
    if s: seconds += int(s.group(1))
    return seconds

if 'video_filename' not in locals():
    print("エラー: セル1で動画をアップロードしてください。")
else:
    !rm -rf /content/extracted_images/*

    real_interval_sec = parse_time_str(撮影間隔_分秒)
    start_offset_sec = parse_time_str(抽出開始_時間_分_秒)
    extract_interval_sec = parse_time_str(抽出間隔_分_秒)

    if real_interval_sec <= 0:
         print("エラー: 撮影間隔は1秒以上に設定してください。")
    else:
        try:
            dt_video_start = datetime.strptime(動画の撮影開始日時.replace("-", "/"), "%Y/%m/%d %H:%M:%S")
            cap = cv2.VideoCapture(video_filename)
            total_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            # 終了時間が指定されていない場合は、動画の最後の時間を終了時間とする
            end_offset_sec = parse_time_str(抽出終了_時間_分_秒)
            if end_offset_sec == 0:
                end_offset_sec = total_video_frames * real_interval_sec
                print(f"抽出終了時間が空欄のため、動画の最後まで抽出します ({end_offset_sec}秒まで)")

            print(f"撮影開始基準: {dt_video_start}")

            extracted_count = 0
            first_image = None
            current_sec = start_offset_sec

            while current_sec <= end_offset_sec:
                target_frame = int(current_sec / real_interval_sec)

                if target_frame >= total_video_frames:
                    break

                cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
                ret, frame = cap.read()
                if not ret: break

                actual_dt = dt_video_start + timedelta(seconds=target_frame * real_interval_sec)
                out_path = f"/content/extracted_images/frame_{actual_dt.strftime('%Y%m%d_%H%M%S')}_{target_frame:06d}.jpg"
                cv2.imwrite(out_path, frame)

                if first_image is None:
                    first_image = frame.copy()

                extracted_count += 1
                current_sec += extract_interval_sec

            cap.release()
            print(f"抽出完了: 計 {extracted_count} 枚の画像を保存しました。")

            if first_image is not None:
                plt.figure(figsize=(8, 6))
                plt.imshow(cv2.cvtColor(first_image, cv2.COLOR_BGR2RGB))
                plt.axis('off')
                plt.show()

        except Exception as e:
            print(f"エラーが発生しました: {e}")

#魚眼補正

撮影したカメラに魚眼レンズが付いている場合は[魚眼補正](https://colab.research.google.com/github/1900690/fisheye-distortion-correction/blob/main/%E9%AD%9A%E7%9C%BC%E8%A3%9C%E6%AD%A3.ipynb)
を行ってDIM / K / Dの値を入手してください

In [ ]:
#@title 3. 画像の魚眼補正（魚眼レンズのみ実行）
import cv2
import numpy as np
import os
import glob
import matplotlib.pyplot as plt

#@markdown **魚眼レンズ 補正パラメータ設定**
#@markdown * ※値はPythonの数式やリストの形式（例: [a, b, c]）で入力してください。
DIM_input = "(640, 480)" #@param {type:"string"}
K_input = "[[500.0, 0.0, 320.0], [0.0, 500.0, 240.0], [0.0, 0.0, 1.0]]" #@param {type:"string"}
D_input = "[-0.1, 0.01, 0.0, 0.0]" #@param {type:"string"}

!rm -rf /content/corrected_images/*
extracted_files = sorted(glob.glob('/content/extracted_images/*.jpg'))

if len(extracted_files) > 0:
    print("魚眼補正を実行しています...")
    try:
        # 入力文字列をPythonオブジェクトおよびnumpy配列に変換
        DIM = eval(DIM_input)  # (width, height) のタプル
        K = np.array(eval(K_input), dtype=np.float64)  # 3x3 カメラ行列
        D = np.array(eval(D_input), dtype=np.float64)  # 歪み係数

        # 補正マップの初期化 (fisheyeモデル用)
        map1, map2 = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), K, DIM, cv2.CV_16SC2)
        first_corrected = None

        for file_path in extracted_files:
            img = cv2.imread(file_path)

            # 画像サイズがDIMと異なる場合はDIMにリサイズ
            if (img.shape[1], img.shape[0]) != DIM:
                img = cv2.resize(img, DIM)

            # 魚眼補正の適用
            undistorted_img = cv2.remap(img, map1, map2, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)

            out_path = os.path.join('/content/corrected_images', os.path.basename(file_path))
            cv2.imwrite(out_path, undistorted_img)

            if first_corrected is None:
                first_corrected = undistorted_img.copy()

        print(f"補正完了: {len(extracted_files)} 枚の画像を処理しました。")

        if first_corrected is not None:
            print("\n【補正画像の1枚目（確認用）】")
            plt.figure(figsize=(8, 6))
            plt.imshow(cv2.cvtColor(first_corrected, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()

    except Exception as e:
        print(f"エラー: パラメータの解析または補正処理に失敗しました。入力形式を確認してください。\n詳細: {e}")
else:
    print("エラー: 補正する画像がありません。先にセル2を実行して画像を抽出してください。")

In [ ]:
#@title 4. 画像をまとめてダウンロード
import shutil
from google.colab import files
import os

#@markdown ダウンロードするZIPファイル名を指定してください。
zip_filename = "extracted_images" #@param {type:"string"}

corrected_dir = '/content/corrected_images'
extracted_dir = '/content/extracted_images'

# セル3で補正が実行され、画像が生成されているか判定
if os.path.exists(corrected_dir) and len(os.listdir(corrected_dir)) > 0:
    target_dir = corrected_dir
    print("魚眼補正済みの画像を圧縮しています...")
else:
    target_dir = extracted_dir
    print("抽出された元画像を圧縮しています（魚眼補正はスキップされました）...")

if not os.path.exists(target_dir) or len(os.listdir(target_dir)) == 0:
    print("エラー: ダウンロードする画像がありません。")
else:
    # フォルダをZIP化
    zip_path = f"/content/{zip_filename}"
    shutil.make_archive(zip_path, 'zip', root_dir=target_dir)

    # ZIPファイルをダウンロード
    print("ダウンロードを開始します...")
    files.download(f"{zip_path}.zip")